# Baselines + Zero-Shot Qwen 2.5 7B — Validation Evaluation

Runs all three baselines on the validation set (726 examples) and saves results for the final comparison.

| Baseline | Method | Time |
|----------|--------|------|
| BM25 | Keyword overlap, no GPU | < 1 min |
| Sentence-Transformer | `all-mpnet-base-v2` cosine sim, CPU | ~3 min |
| Zero-shot Qwen 2.5 7B | Untuned model, same prompt | ~1.5 hr (T4) / ~25 min (A100) |

**Ground truth:** teacher scores from DeepSeek (the `score` field in each validation example)

## 0. Check GPU

In [13]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Wed Jun  3 21:52:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             41W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Install

In [14]:
!pip install rank_bm25 sentence-transformers scipy
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install --no-deps xformers 'trl<0.9.0' peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-b6lbhjny/unsloth_9fca0cb68e4740f99abc715da15f8602
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-b6lbhjny/unsloth_9fca0cb68e4740f99abc715da15f8602
  Resolved https://github.com/unslothai/unsloth.git to commit b0572bd233efc120d50a940f13af89eb879d0dd0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.0 MB/s eta 0:00:00
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


In [15]:
import json, os, time, collections
import numpy as np
import torch
from scipy.stats import pearsonr, spearmanr
print('Imports OK')

Imports OK


## 2. Load Validation Data

In [16]:
!git clone https://github.com/eka026/fit-my-resume.git /content/repo
VAL_PATH = '/content/repo/data/instruction_tuning/instruction_tuning_validation.jsonl'

fatal: destination path '/content/repo' already exists and is not an empty directory.


In [17]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

val_data = load_jsonl(VAL_PATH)
print(f'Validation examples: {len(val_data)}')

# Parse out the fields we need
examples = []
for ex in val_data:
    parts  = ex['input'].split('JOB_DESCRIPTION:')
    resume = parts[0].replace('RESUME:', '').strip()
    job    = parts[1].strip()
    examples.append({
        'pair_id':       ex['metadata']['pair_id'],
        'strategy':      ex['metadata']['pairing_strategy'],
        'resume':        resume,
        'job':           job,
        'input':         ex['input'],
        'teacher_score': json.loads(ex['output'])['score'],
    })

teacher_scores = [ex['teacher_score'] for ex in examples]
print(f'Teacher score — mean: {np.mean(teacher_scores):.1f}, range: {min(teacher_scores)}–{max(teacher_scores)}')

Validation examples: 726
Teacher score — mean: 21.8, range: 0–88


## 3. Baseline 1 — BM25

Keyword overlap between the resume and job description. No GPU, runs in seconds.

In [18]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

corpus = [tokenize(ex['resume']) for ex in examples]
bm25   = BM25Okapi(corpus)

bm25_scores = []
for i, ex in enumerate(examples):
    scores = bm25.get_scores(tokenize(ex['job']))
    bm25_scores.append(float(scores[i]))

pearson_bm25,  _ = pearsonr(teacher_scores, bm25_scores)
spearman_bm25, _ = spearmanr(teacher_scores, bm25_scores)
print(f'BM25 — Pearson: {pearson_bm25:.3f}  Spearman: {spearman_bm25:.3f}')

BM25 — Pearson: 0.110  Spearman: 0.126


## 4. Baseline 2 — Sentence-Transformer

Cosine similarity of `all-mpnet-base-v2` embeddings, scaled to 0–100. Runs on CPU to keep GPU free for Qwen.

In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device='cpu')

print('Encoding resumes...')
resume_embs = st_model.encode([ex['resume'] for ex in examples], batch_size=32, show_progress_bar=True)
print('Encoding job descriptions...')
job_embs    = st_model.encode([ex['job']    for ex in examples], batch_size=32, show_progress_bar=True)

st_scores = [
    float(max(0.0, min(100.0, cosine_similarity(resume_embs[i:i+1], job_embs[i:i+1])[0][0] * 100)))
    for i in range(len(examples))
]

pearson_st,  _ = pearsonr(teacher_scores, st_scores)
spearman_st, _ = spearmanr(teacher_scores, st_scores)
mae_st = np.mean(np.abs(np.array(teacher_scores) - np.array(st_scores)))
print(f'Sentence-Transformer — Pearson: {pearson_st:.3f}  Spearman: {spearman_st:.3f}  MAE: {mae_st:.1f}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding resumes...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Encoding job descriptions...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Sentence-Transformer — Pearson: 0.535  Spearman: 0.521  MAE: 21.0


## 5. Load Qwen 2.5 7B (4-bit, untuned)

In [23]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = 4096,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print('Model loaded!')

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded!


## 6. Zero-Shot Inference on Full Validation Set

In [28]:
SYSTEM_PROMPT = (
    'You are a professional resume evaluation assistant. '
    'Evaluate the resume against the job description and return ONLY valid JSON '
    'with exactly this structure, no markdown, no extra text:\n'
    '{"score": <integer 0-100>, "explanation": "<string>", '
    '"matched_qualifications": ["<string>", ...], '
    '"missing_qualifications": ["<string>", ...], '
    '"resume_suggestions": [{"section": "<string>", "action": "<string>", '
    '"suggestion": "<string>", "evidence_from_resume": "<string>"}]}'
)

def run_inference(input_text, max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': input_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids      = inputs,
            max_new_tokens  = max_new_tokens,
            temperature     = 0.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

In [30]:
response = run_inference(examples[0]['input'])
clean = response.replace('```json', '').replace('```', '').strip()
print(repr(clean[:800]))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'{"score": 45, "explanation": "The candidate has extensive experience in construction management and safety, but the job description requires specific skills in construction project management, which the candidate lacks. The candidate\'s experience is more aligned with industrial maintenance and safety rather than construction project management.", "matched_qualifications": ["Safety Standards", "Site Safety / Safety Standards", "Leadership Skills", "Self Motivated", "Team Building", "Team Player"], "missing_qualifications": ["Construction Project Management", "Schedule Monitoring", "Project Financials", "Coordinating Activities of External Vendors", "Interface with Real Estate Design Permitting Facilities IT Operations Training Functions as Well as Landlords and External AHJs", "Work with th'


In [31]:
import re

qwen_results = []
parse_errors = 0
start = time.time()

for i, ex in enumerate(examples):
    response = run_inference(ex['input'], max_new_tokens=80)
    try:
        match = re.search(r'"score"\s*:\s*(\d+)', response)
        pred_score = int(match.group(1)) if match else None
        if pred_score is None:
            parse_errors += 1
    except Exception:
        pred_score = None
        parse_errors += 1

    qwen_results.append({
        'pair_id':    ex['pair_id'],
        'strategy':   ex['strategy'],
        'gt_score':   ex['teacher_score'],
        'pred_score': pred_score,
        'raw_output': response,
    })

    if (i + 1) % 50 == 0 or i == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(examples) - i - 1) / rate / 60
        print(f'[{i+1}/{len(examples)}]  {elapsed/60:.1f} min elapsed  ~{remaining:.0f} min left  '
              f'parse ok: {i+1-parse_errors}/{i+1}')

total_min = (time.time() - start) / 60
print(f'\nDone in {total_min:.1f} min  |  Parse errors: {parse_errors}/{len(examples)}')

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=80

[1/726]  0.1 min elapsed  ~41 min left  parse ok: 1/1


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

[50/726]  2.7 min elapsed  ~37 min left  parse ok: 50/50


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

...

[650/726]  35.3 min elapsed  ~4 min left  parse ok: 649/650


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

[700/726]  38.1 min elapsed  ~1 min left  parse ok: 699/700


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Done in 39.5 min  |  Parse errors: 1/726


## 7. Save Results

In [32]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/fit-my-resume/results'
os.makedirs(OUT_DIR, exist_ok=True)

# Save zero-shot Qwen predictions
QWEN_PATH = f'{OUT_DIR}/zero_shot_qwen_val_outputs.jsonl'
with open(QWEN_PATH, 'w', encoding='utf-8') as f:
    for row in qwen_results:
        f.write(json.dumps(row) + '\n')
print(f'Saved Qwen outputs: {QWEN_PATH}')

# Also download as backup
LOCAL_PATH = '/content/zero_shot_qwen_val_outputs.jsonl'
with open(LOCAL_PATH, 'w', encoding='utf-8') as f:
    for row in qwen_results:
        f.write(json.dumps(row) + '\n')
from google.colab import files
files.download(LOCAL_PATH)

Mounted at /content/drive
Saved Qwen outputs: /content/drive/MyDrive/fit-my-resume/results/zero_shot_qwen_val_outputs.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Results — All Baselines

In [33]:
# Zero-shot Qwen metrics
valid_qwen = [(r['gt_score'], r['pred_score']) for r in qwen_results if r['pred_score'] is not None]
qwen_gt   = [v[0] for v in valid_qwen]
qwen_pred = [v[1] for v in valid_qwen]
pearson_qwen,  _ = pearsonr(qwen_gt, qwen_pred)
spearman_qwen, _ = spearmanr(qwen_gt, qwen_pred)
mae_qwen = np.mean(np.abs(np.array(qwen_gt) - np.array(qwen_pred)))
parse_rate = 100 * len(valid_qwen) / len(qwen_results)

print('=' * 68)
print(f'{"Baseline":<26} {"Pearson":>9} {"Spearman":>10} {"MAE":>7} {"Parse%":>8}')
print('-' * 68)
print(f'{"BM25":<26} {pearson_bm25:>9.3f} {spearman_bm25:>10.3f} {"—":>7} {"—":>8}')
print(f'{"Sentence-Transformer":<26} {pearson_st:>9.3f} {spearman_st:>10.3f} {mae_st:>7.1f} {"—":>8}')
print(f'{"Zero-shot Qwen 2.5 7B":<26} {pearson_qwen:>9.3f} {spearman_qwen:>10.3f} {mae_qwen:>7.1f} {parse_rate:>7.1f}%')
print('=' * 68)
print('(Fine-tuned Qwen row will be added by Enes after Aliya shares results)')

Baseline                     Pearson   Spearman     MAE   Parse%
--------------------------------------------------------------------
BM25                           0.110      0.126       —        —
Sentence-Transformer           0.535      0.521    21.0        —
Zero-shot Qwen 2.5 7B          0.563      0.591    20.2    99.9%
(Fine-tuned Qwen row will be added by Enes after Aliya shares results)


In [34]:
# Per-strategy breakdown for zero-shot Qwen
strat_groups = collections.defaultdict(list)
for r in qwen_results:
    if r['pred_score'] is not None:
        strat_groups[r['strategy']].append((r['gt_score'], r['pred_score']))

print('Zero-shot Qwen — per pairing strategy:')
print(f'{"Strategy":<22} {"n":>5} {"MAE":>8} {"Spearman":>10}')
print('-' * 50)
for strat, pairs in sorted(strat_groups.items()):
    g = [p[0] for p in pairs]
    p = [p[1] for p in pairs]
    spr, _ = spearmanr(g, p)
    m = np.mean(np.abs(np.array(g) - np.array(p)))
    print(f'{strat:<22} {len(pairs):>5} {m:>8.1f} {spr:>10.3f}')

Zero-shot Qwen — per pairing strategy:
Strategy                   n      MAE   Spearman
--------------------------------------------------
medium_tfidf             238     23.0      0.412
strong_hybrid            243     20.5      0.607
weak_random              244     17.2      0.345


In [35]:
# Sanity check — sample predictions
print(f'{"pair_id":<45} {"GT":>4} {"Pred":>6}')
print('-' * 60)
for r in qwen_results[:15]:
    pred_str = str(r['pred_score']) if r['pred_score'] is not None else 'ERR'
    print(f'{r["pair_id"][:44]:<45} {r["gt_score"]:>4} {pred_str:>6}')

pair_id                                         GT   Pred
------------------------------------------------------------
validation_10149490_job_000176_strong_hybrid    35     45
validation_10149490_job_000253_medium_tfidf     10     30
validation_10149490_job_000772_weak_random      10     30
validation_95085510_job_000274_strong_hybrid    25     45
validation_95085510_job_000765_medium_tfidf     15     55
validation_95085510_job_000406_weak_random      25     55
validation_26585242_job_000094_strong_hybrid    55     55
validation_26585242_job_000697_medium_tfidf     20     55
validation_26585242_job_000624_weak_random      15      2
validation_13520837_job_000643_strong_hybrid    35     45
validation_13520837_job_000773_medium_tfidf     35     75
validation_13520837_job_000157_weak_random      15     40
validation_15973307_job_000599_strong_hybrid    55     65
validation_15973307_job_000687_medium_tfidf      5     45
validation_15973307_job_000217_weak_random       5     30
